<a href="https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vaishnavikabbe/AIML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/vaishnavikabbe/AIML.git"
REPO_DIR = "/content/AIML"

if not os.path.exists(REPO_DIR):
    print("Cloning AIML repository...")
    subprocess.run(
        ["git", "clone", REPO_URL, REPO_DIR],
        check=True
    )
    print("Repository cloned successfully.")
else:
    print("AIML repository already exists.")

print("\nRepository contents:")
print(os.listdir(REPO_DIR))

AIML repository already exists.

Repository contents:
['DATA_USE.md', 'README.md', 'notebooks', 'docs', 'work', 'SETUP.md', 'data', 'scripts', 'skills', 'outputs', '.git', 'AGENTS.md', 'LICENSE', '.github', 'submission', 'CLAUDE.md', 'GUIDE.md', '.gitignore', 'requirements.txt']


In [16]:
import os

DATA_PATH = "/content/AIML/data/raw/content_refresh_anonymized.csv"

if os.path.exists(DATA_PATH):
    print("Dataset found!")
    print(DATA_PATH)
else:
    print("Dataset NOT found.")
    print("Checking data folders...")

    for root, dirs, files in os.walk("/content/AIML"):
        for file in files:
            if "content_refresh" in file.lower():
                print("Found:", os.path.join(root, file))

Dataset found!
/content/AIML/data/raw/content_refresh_anonymized.csv


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Two paper findings and methodology questions

**Finding 1:** The FlyRank research uses search-related outcomes to evaluate whether certain signals are associated with better search performance. My methodology question is how the labels or outcomes were constructed and whether they represent a consistent observed outcome across the dataset. I would want to confirm that the label is available independently of the features used for prediction.

**Finding 2:** The research shows that some content and search signals can be useful for ranking or prioritizing pages. My methodology question is whether the validation design reflects the way the model would be used in practice. In particular, I would check whether the evaluation avoids information from the future and whether similar pages or entities can appear in both training and test data.

Overall, I treat the paper findings as useful evidence to investigate rather than as causal proof. My own analysis will use observed, measured, and directional results.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Record the two methodology checks I will apply to my own analysis.

methodology_checks = [
    "Check that the target is based on an observed outcome and is not derived from the input features.",
    "Check that the validation split avoids future information and other forms of leakage."
]

for i, check in enumerate(methodology_checks, 1):
    print(f"Check {i}: {check}")

Check 1: Check that the target is based on an observed outcome and is not derived from the input features.
Check 2: Check that the validation split avoids future information and other forms of leakage.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Honest validation split

My model should be evaluated on data that represents pages it would encounter after training. I will use a time-aware split when a valid time field is available. This is more realistic than randomly mixing observations because information from a later period should not be used to evaluate a decision that would have been made earlier.

I will compare the model's metric with the Week-5 baseline using the same target and evaluation metric. The difference between the two results will be treated as measured evidence of performance under this split, not as proof that the model will perform the same way on future data.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

DATA_PATH = "/content/AIML/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

possible_time_cols = [
    col for col in df.columns
    if any(word in col.lower() for word in ["date", "time", "month", "period"])
]

print("\nPossible date/time columns:")
print(possible_time_cols)

Dataset loaded successfully.
Rows: 30000
Columns: 44

Possible date/time columns:
['days_since_last_update']


In [19]:
import pandas as pd

DATA_PATH = "/content/AIML/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

possible_time_cols = [
    col for col in df.columns
    if any(word in col.lower() for word in ["date", "time", "month", "period"])
]

print("\nPossible date/time columns:")
print(possible_time_cols)

Dataset loaded successfully.
Rows: 30000
Columns: 44

Possible date/time columns:
['days_since_last_update']


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check all columns for possible leakage

print("All dataset columns:")
for col in df.columns:
    print(col)

risk_words = [
    "target",
    "label",
    "trend",
    "future",
    "next",
    "outcome"
]

possible_leakage = [
    col for col in df.columns
    if any(word in col.lower() for word in risk_words)
]

print("\nColumns requiring leakage review:")
print(possible_leakage)

All dataset columns:
content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct

Columns requiring leakage review:
['trend_direction', 'trend_pct']


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite

My analysis does not prove that a particular content signal causes better search performance.

A safer claim is:

The analysis provides observed and measured evidence about whether search-performance and content signals can help prioritize pages for content refresh review. The results are directional decision-support and should not be interpreted as causal proof or a guarantee of future search performance.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
final_claim = (
    "Observed and measured results provide directional decision-support "
    "for prioritizing pages for content refresh review. The analysis does "
    "not establish causation or guarantee future search performance."
)

print("Final research claim:")
print(final_claim)

Final research claim:
Observed and measured results provide directional decision-support for prioritizing pages for content refresh review. The analysis does not establish causation or guarantee future search performance.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.